# EEG Feature Analysis: Actual vs Estimated Signal
## Reviewer Response — Feature Comparison

**Addresses:**
- Reviewer 1 Point 1: Volume Conduction (prove estimated signal carries meaningful neural info)
- Reviewer 1 Point 4: Temporal and Spectral Consistency (PSD comparison)

**Analysis:**
Compares actual and ANN-estimated signals across three feature domains:
1. **Frequency domain** — Band power (delta, theta, alpha, beta, gamma) via Welch PSD
2. **Time domain** — Hjorth parameters (Activity, Mobility, Complexity)
3. **Wavelet domain** — Wavelet energy per decomposition level (db4, 5 levels)

**Source:** Within-subject LOTO cache pkl files (all 40 subjects)
**Model:** ANN_MLP (best-performing model)


In [ ]:
# =============================================================================
# 0. IMPORTS & CONFIG
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

import pywt
from scipy.signal import welch

# ── Sesuaikan path ─────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/Thesis/channel_estimation'

# ── Config ─────────────────────────────────────────────────────────────────────
FS          = 128
NPERSEG     = 256
TARGET_CHS  = ['FZ', 'FC2', 'CZ', 'CP2', 'P4']
BEST_MODEL  = 'ANN_MLP'
TASKS       = ['Arithmetic', 'Mirror_image', 'Stroop']
ALL_SUBJECTS = list(range(1, 41))
MODES       = ['original', 'derivative']

BANDS = {
    'Delta' : (0.5,  4),
    'Theta' : (4,    8),
    'Alpha' : (8,   13),
    'Beta'  : (14,  30),
    'Gamma' : (30,  50),
}

FEAT_DIR = os.path.join(OUTPUT_DIR, 'feature_comparison')
os.makedirs(FEAT_DIR, exist_ok=True)

print("Config loaded.")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"Target channels : {TARGET_CHS}")
print(f"Best model      : {BEST_MODEL}")

## 1. Feature Extraction Functions

In [ ]:
# ── Band Power ────────────────────────────────────────────────────────────────
def band_power(freqs, psd, fmin, fmax):
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    return np.trapz(psd[idx], freqs[idx])

def extract_psd_features(signal):
    """Band power per EEG frequency band via Welch method."""
    freqs, psd = welch(signal, fs=FS, nperseg=NPERSEG)
    features   = {f'BP_{band}': band_power(freqs, psd, fmin, fmax)
                  for band, (fmin, fmax) in BANDS.items()}
    return features, freqs, psd

# ── Hjorth Parameters ─────────────────────────────────────────────────────────
def extract_hjorth(signal):
    """Hjorth Activity, Mobility, Complexity."""
    activity   = np.var(signal)
    d1         = np.diff(signal)
    d2         = np.diff(d1)
    mobility   = (np.sqrt(np.var(d1) / activity)
                  if activity > 0 else 0)
    complexity = (np.sqrt(np.var(d2) / np.var(d1)) / mobility
                  if (mobility > 0 and np.var(d1) > 0) else 0)
    return {
        'Hjorth_Activity'  : activity,
        'Hjorth_Mobility'  : mobility,
        'Hjorth_Complexity': complexity,
    }

# ── Wavelet Energy ────────────────────────────────────────────────────────────
def extract_wavelet_energy(signal, wavelet='db4', level=5):
    """Wavelet energy per decomposition level using db4."""
    coeffs   = pywt.wavedec(signal, wavelet, level=level)
    features = {}
    for i, coeff in enumerate(coeffs):
        label = 'cA' if i == 0 else f'cD{level - i + 1}'
        features[f'WE_{label}'] = np.sum(coeff ** 2)
    return features

# ── Combined ──────────────────────────────────────────────────────────────────
def extract_all_features(signal):
    psd_feats, freqs, psd = extract_psd_features(signal)
    hjorth_feats           = extract_hjorth(signal)
    wavelet_feats          = extract_wavelet_energy(signal)
    return {**psd_feats, **hjorth_feats, **wavelet_feats}, freqs, psd

print("Feature extraction functions defined.")

## 2. Load Signals from Cache and Extract Features

In [ ]:
def collect_features_from_cache(subject_ids, mode):
    """
    Load actual dan estimated signals dari cache pkl.
    Extract fitur untuk setiap segment.
    """
    MODE_LABEL = {'original': 'Asli', 'derivative': 'Turunan'}
    records    = []

    for sub_id in subject_ids:
        cache_path = os.path.join(
            OUTPUT_DIR,
            f"training_cache_{MODE_LABEL[mode]}_sub{sub_id}.pkl")

        if not os.path.exists(cache_path):
            print(f"  [SKIP] Subject {sub_id} not found")
            continue

        with open(cache_path, 'rb') as f:
            cache = pickle.load(f)

        for task in TASKS:
            if task not in cache:
                continue
            for split_key, channels in cache[task].items():
                for ch in TARGET_CHS:
                    if ch not in channels:
                        continue
                    ch_data = channels[ch]
                    if ('y_test_orig' not in ch_data or
                            BEST_MODEL not in ch_data.get('predictions', {})):
                        continue

                    sig_actual = ch_data['y_test_orig']
                    sig_estim  = ch_data['predictions'][BEST_MODEL]

                    for sig, sig_type in [(sig_actual, 'Actual'),
                                          (sig_estim,  'Estimated')]:
                        feats, _, _ = extract_all_features(sig)
                        records.append({
                            'Subject'    : sub_id,
                            'Task'       : task,
                            'Split'      : split_key,
                            'Channel'    : ch,
                            'Input_Mode' : mode,
                            'Signal_Type': sig_type,
                            **feats
                        })

    df = pd.DataFrame(records)
    print(f"  Extracted {len(df)} records for mode={mode}")
    return df

print("collect_features_from_cache() defined.")

## 3. Compute Feature Difference (%)

In [ ]:
def compute_feature_diff(df, mode):
    """
    Pivot actual vs estimated.
    Compute percentage difference per feature.
    """
    feat_cols  = [c for c in df.columns
                  if c.startswith(('BP_', 'Hjorth_', 'WE_'))]
    merge_keys = ['Subject', 'Task', 'Split', 'Channel', 'Input_Mode']

    actual = df[df['Signal_Type'] == 'Actual'].copy()
    estim  = df[df['Signal_Type'] == 'Estimated'].copy()
    merged = actual.merge(estim, on=merge_keys, suffixes=('_actual', '_estim'))

    for feat in feat_cols:
        col_a = f'{feat}_actual'
        col_e = f'{feat}_estim'
        merged[f'{feat}_diff_pct'] = (
            np.abs(merged[col_e] - merged[col_a])
            / (np.abs(merged[col_a]) + 1e-10) * 100
        )

    return merged

print("compute_feature_diff() defined.")

## 4. PSD Comparison Plot

In [ ]:
def plot_psd_comparison(mode, save_dir):
    """
    Plot PSD actual vs estimated per target channel.
    Averaged across all subjects, tasks, splits.
    """
    MODE_LABEL  = {'original': 'Actual Signal',
                   'derivative': 'Actual + Derivative Signal'}
    MODE_FILE   = {'original': 'Asli', 'derivative': 'Turunan'}
    band_colors = {
        'Delta': '#E8F4FD', 'Theta': '#D5E8D4',
        'Alpha': '#FFF2CC', 'Beta' : '#F8CECC',
        'Gamma': '#E1D5E7'
    }

    fig, axes = plt.subplots(1, len(TARGET_CHS),
                              figsize=(5*len(TARGET_CHS), 4))

    for ax, ch in zip(axes, TARGET_CHS):
        ch_actual, ch_estim = [], []

        for sub_id in ALL_SUBJECTS:
            cache_path = os.path.join(
                OUTPUT_DIR,
                f"training_cache_{MODE_FILE[mode]}_sub{sub_id}.pkl")
            if not os.path.exists(cache_path):
                continue
            with open(cache_path, 'rb') as f:
                cache = pickle.load(f)
            for task in TASKS:
                if task not in cache:
                    continue
                for split_key, channels in cache[task].items():
                    if ch not in channels:
                        continue
                    cd = channels[ch]
                    if ('y_test_orig' not in cd or
                            BEST_MODEL not in cd.get('predictions', {})):
                        continue
                    ch_actual.append(cd['y_test_orig'])
                    ch_estim.append(cd['predictions'][BEST_MODEL])

        if not ch_actual:
            ax.set_title(f"{ch}\n(no data)")
            continue

        sig_a  = np.concatenate(ch_actual)
        sig_e  = np.concatenate(ch_estim)
        fa, pa = welch(sig_a, fs=FS, nperseg=NPERSEG)
        fe, pe = welch(sig_e, fs=FS, nperseg=NPERSEG)

        idx = fa <= 50
        ax.semilogy(fa[idx], pa[idx], color='black',
                    lw=1.5, label='Actual')
        ax.semilogy(fe[idx], pe[idx], color='steelblue',
                    lw=1.5, linestyle='--', label='Estimated')

        for band, (fmin, fmax) in BANDS.items():
            ax.axvspan(fmin, fmax, alpha=0.15,
                       color=band_colors[band], label=band)

        ax.set_title(f"Channel {ch}", fontsize=11, fontweight='bold')
        ax.set_xlabel("Frequency (Hz)", fontsize=9)
        ax.set_ylabel("PSD (µV²/Hz)", fontsize=9)
        ax.set_xlim([0, 50])
        ax.grid(True, alpha=0.3)

        if ch == TARGET_CHS[0]:
            h, l = ax.get_legend_handles_labels()
            ax.legend(h[:2], l[:2], fontsize=8, loc='upper right')

    fig.suptitle(
        f"PSD Comparison: Actual vs Estimated | "
        f"{MODE_LABEL[mode]} | {BEST_MODEL} | N=40 subjects",
        fontsize=12, fontweight='bold')
    plt.tight_layout()

    fname = os.path.join(save_dir, f"psd_comparison_{mode}.png")
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  [SAVED] {fname}")

print("plot_psd_comparison() defined.")

## 5. Summary Table Builder

In [ ]:
def make_summary_table(df_diff, mode):
    """Mean feature difference % per channel, grouped by domain."""
    diff_cols = [c for c in df_diff.columns if c.endswith('_diff_pct')]

    summary = (
        df_diff[df_diff['Input_Mode'] == mode]
        .groupby('Channel')[diff_cols]
        .mean()
        .round(2)
        .reset_index()
    )

    rename = {c: c.replace('_diff_pct', '') for c in diff_cols}
    summary = summary.rename(columns=rename)

    ch_order = ['FZ', 'FC2', 'CZ', 'CP2', 'P4']
    summary['Channel'] = pd.Categorical(
        summary['Channel'], categories=ch_order, ordered=True)
    summary = summary.sort_values('Channel').reset_index(drop=True)
    return summary

print("make_summary_table() defined.")

## 6. Run Full Analysis

In [ ]:
all_diffs = []

for mode in MODES:
    print(f"\n{'='*55}")
    print(f"  Mode: {mode.upper()}")
    print(f"{'='*55}")

    # ── Extract features ───────────────────────────────────────────────
    print("  Extracting features from cache...")
    df_feats = collect_features_from_cache(ALL_SUBJECTS, mode)

    # ── Compute diff ───────────────────────────────────────────────────
    df_diff = compute_feature_diff(df_feats, mode)
    all_diffs.append(df_diff)

    # ── PSD plot ───────────────────────────────────────────────────────
    print("  Generating PSD comparison plot...")
    plot_psd_comparison(mode, FEAT_DIR)

    # ── Summary table ──────────────────────────────────────────────────
    summary = make_summary_table(df_diff, mode)
    out_path = os.path.join(FEAT_DIR, f"feature_diff_{mode}.csv")
    summary.to_csv(out_path, index=False, encoding='utf-8-sig')

    # ── Print key results ──────────────────────────────────────────────
    bp_cols = [c for c in summary.columns if c.startswith('BP_')]
    hj_cols = [c for c in summary.columns if c.startswith('Hjorth_')]
    we_cols = [c for c in summary.columns if c.startswith('WE_')]

    print(f"\n  Band Power Diff % (N=40 subjects, mean):")
    print(summary[['Channel'] + bp_cols].to_string(index=False))

    print(f"\n  Hjorth Diff % (N=40 subjects, mean):")
    print(summary[['Channel'] + hj_cols].to_string(index=False))

    print(f"\n  Wavelet Energy Diff % (N=40 subjects, mean):")
    print(summary[['Channel'] + we_cols].to_string(index=False))

# Save all diffs
df_all_diff = pd.concat(all_diffs, ignore_index=True)
df_all_diff.to_csv(
    os.path.join(FEAT_DIR, 'feature_diff_all.csv'),
    index=False, encoding='utf-8-sig')

print(f"\nAll results saved to: {FEAT_DIR}")

## 7. Visualization — Feature Diff Heatmap

In [ ]:
def plot_feature_diff_heatmap(mode):
    """Heatmap of feature difference % per channel per feature."""
    summary = pd.read_csv(
        os.path.join(FEAT_DIR, f"feature_diff_{mode}.csv"))

    feat_cols = [c for c in summary.columns if c != 'Channel']
    data      = summary.set_index('Channel')[feat_cols]

    fig, ax = plt.subplots(figsize=(max(12, len(feat_cols)*0.7), 5))
    im = ax.imshow(data.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=20)

    ax.set_xticks(range(len(feat_cols)))
    ax.set_xticklabels(feat_cols, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(data.index, fontsize=10)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.values[i, j]
            ax.text(j, i, f"{val:.1f}", ha='center', va='center',
                    fontsize=7,
                    color='white' if val > 12 else 'black')

    plt.colorbar(im, ax=ax, label='Mean Difference (%)')
    MODE_LABEL = {'original': 'Actual Signal',
                  'derivative': 'Actual + Derivative Signal'}
    ax.set_title(
        f"Feature Difference: Actual vs Estimated | "
        f"{MODE_LABEL[mode]} | {BEST_MODEL}",
        fontsize=11, fontweight='bold')
    plt.tight_layout()

    fname = os.path.join(FEAT_DIR, f"feature_heatmap_{mode}.png")
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[SAVED] {fname}")

for mode in MODES:
    plot_feature_diff_heatmap(mode)

## Notes

- **PSD figures** are saved as `psd_comparison_original.png` and `psd_comparison_derivative.png`
- **Feature diff CSV** per mode: `feature_diff_original.csv`, `feature_diff_derivative.csv`
- **Heatmap** shows percentage difference per feature per channel — values close to 0 indicate estimated signal preserves the feature well
- Results from this notebook fill the placeholders in Response Letter R1.1 (Volume Conduction) and R1.4 (PSD Analysis)
- A small feature difference across all domains supports the argument that estimated signals carry meaningful neural information beyond passive volume conduction
